In [6]:
import asyncio
from pysnmp.hlapi.v3arch.asyncio import (
    SnmpEngine, CommunityData, UdpTransportTarget, ContextData,
    ObjectType, ObjectIdentity, get_cmd
)

async def _get_oid_v2v1(oid: str, target: str, version: str, community: str = "public"):
    errorIndication, errorStatus, errorIndex, varBinds = await get_cmd(
        SnmpEngine(),
        CommunityData(community, mpModel=0 if version == "v1" else 1),
        await UdpTransportTarget.create((target, 161)),
        ContextData(),
        ObjectType(ObjectIdentity(oid))
    )

    if errorIndication:
        return f"Error: {errorIndication}"
    elif errorStatus:
        return f"Error: {errorStatus.prettyPrint()}"
    else:
        # single OID => single varBind
        return str(varBinds[0][1])

async def main():
    R1 = "192.168.100.2"
    OIDS = {
        "Contact": ".1.3.6.1.2.1.1.4.0",
        "Name": ".1.3.6.1.2.1.1.5.0",
        "Location": ".1.3.6.1.2.1.1.6.0",
        "IfNumber": ".1.3.6.1.2.1.2.1.0",
        "Uptime": ".1.3.6.1.2.1.1.3.0",
    }

    v1_results = {}
    for key, oid in OIDS.items():
        v1_results[key] = await _get_oid_v2v1(oid, R1, version="v2c", community="lab123")

    print(v1_results)

if __name__ == "__main__":
    await main()


{'Contact': '', 'Name': 'R1', 'Location': '', 'IfNumber': '12', 'Uptime': '750188'}
